# SQL query from table names - Continued

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [8]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [28]:
context = [ {'role':'system', 'content':"""
 CREATE SEVERAL (3+) TABLES HERE
"""} ]



In [30]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
WRITE IN YOUR CONTEXT QUERIES HERE
"""
})

In [32]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [35]:
#new
context_user = context.copy()
print(return_CCRMSQL("""YOUR QUERY HERE""", context_user))

```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_Usr = s.ID_Usr
ORDER BY s.salary DESC
LIMIT 1;
```


In [37]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("YOUR QUERY HERE", old_context_user))

This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;
```

This SQL query retrieves the name of the employee who is best paid by joining the "employees" table with the "salary" table on the ID_usr field. It then orders the result by salary in descending order and limits the output to only one row, which corresponds to the employee with the highest salary.


In [38]:
#new
print(return_CCRMSQL("YOUR QUERY HERE", context_user))

```sql
SELECT st.Institution, AVG(sa.salary) AS avg_salary
FROM studies st
JOIN employees e ON st.ID_Usr = e.ID_Usr
JOIN salary sa ON e.ID_Usr = sa.ID_Usr
GROUP BY st.Institution
ORDER BY avg_salary DESC
LIMIT 1;
```


In [39]:
#old
print(return_CCRMSQL("YOUR QUERY HERE", old_context_user))

This is your SQL:
```sql
SELECT s.Institution
FROM studies s
JOIN salary sa ON s.ID_usr = sa.ID_usr
GROUP BY s.Institution
ORDER BY AVG(sa.salary) DESC
LIMIT 1;
```

This SQL query joins the "studies" and "salary" tables on the ID_usr column. It then calculates the average salary for each institution, orders the results in descending order based on the average salary, and returns the institution with the highest average salary.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv

# Load API key
_ = load_dotenv(find_dotenv())
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Function to call the model
def return_CCRMSQL(user_message, context):
    client = OpenAI(api_key=OPENAI_API_KEY)

    newcontext = context.copy()
    newcontext.append({
        "role": "user",
        "content": "question: " + user_message
    })

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=newcontext,
        temperature=0,
    )

    return response.choices[0].message.content


# ============================================================
# OLD PROMPT
# ============================================================

old_context = [
    {
        "role": "system",
        "content": """
You are a bot to assist in creating SQL commands.
All your answers should start with: This is your SQL:
After that, write the SQL query that can do what the user requests.
Your database is composed by a SQL database with some tables.
Try to maintain the SQL order simple and efficient.
"""
    }
]

old_context.append({
    "role": "system",
    "content": """
first table:
{
    "tableName": "employees",
    "fields": [
        {
            "name": "ID_usr",
            "type": "int"
        },
        {
            "name": "name",
            "type": "varchar"
        }
    ]
}
"""
})

old_context.append({
    "role": "system",
    "content": """
second table:
{
    "tableName": "salary",
    "fields": [
        {
            "name": "ID_usr",
            "type": "int"
        },
        {
            "name": "year",
            "type": "date"
        },
        {
            "name": "salary",
            "type": "float"
        }
    ]
}
"""
})

old_context.append({
    "role": "system",
    "content": """
third table:
{
    "tableName": "studies",
    "fields": [
        {
            "name": "ID",
            "type": "int"
        },
        {
            "name": "ID_usr",
            "type": "int"
        },
        {
            "name": "educational_level",
            "type": "int"
        },
        {
            "name": "Institution",
            "type": "varchar"
        },
        {
            "name": "Years",
            "type": "date"
        },
        {
            "name": "Speciality",
            "type": "varchar"
        }
    ]
}
"""
})


# ============================================================
# NEW PROMPT
# ============================================================

context = [
    {
        "role": "system",
        "content": """
CREATE TABLE employees (
    ID_usr INTEGER,
    name VARCHAR
);

CREATE TABLE salary (
    ID_usr INTEGER,
    year DATE,
    salary FLOAT
);

CREATE TABLE studies (
    ID INTEGER,
    ID_usr INTEGER,
    educational_level INTEGER,
    Institution VARCHAR,
    Years DATE,
    Speciality VARCHAR
);
"""
    }
]

# FEW SHOT SAMPLES
context.append({
    "role": "system",
    "content": """
-- Maintain the SQL order simple and efficient as you can, using valid SQL Lite.
-- Answer only with the SQL query.

Question: Who is the best paid employee?
SQL:
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;

Question: What is the average salary by institution?
SQL:
SELECT st.Institution, AVG(sa.salary) AS avg_salary
FROM studies st
JOIN salary sa ON st.ID_usr = sa.ID_usr
GROUP BY st.Institution
ORDER BY avg_salary DESC;

Question: List the employees and their specialities.
SQL:
SELECT e.name, st.Speciality
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr;

Question: Which employees have a salary higher than 40000?
SQL:
SELECT e.name, s.salary
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.salary > 40000;
"""
})


# ============================================================
# NL2SQL SAMPLES — SAME STYLE AS NOTEBOOK
# ============================================================

question_1 = "Who is the best paid employee?"
question_2 = "What is the average salary by institution?"
question_3 = "List the employees and their specialities."
question_4 = "Which employees have a salary higher than 40000?"


print("=" * 80)
print("QUESTION 1:", question_1)
print("=" * 80)

# new
context_user = context.copy()
print("\nNEW PROMPT OUTPUT:")
print(return_CCRMSQL(question_1, context_user))

# old
old_context_user = old_context.copy()
print("\nOLD PROMPT OUTPUT:")
print(return_CCRMSQL(question_1, old_context_user))


print("\n" + "=" * 80)
print("QUESTION 2:", question_2)
print("=" * 80)

# new
context_user = context.copy()
print("\nNEW PROMPT OUTPUT:")
print(return_CCRMSQL(question_2, context_user))

# old
old_context_user = old_context.copy()
print("\nOLD PROMPT OUTPUT:")
print(return_CCRMSQL(question_2, old_context_user))


print("\n" + "=" * 80)
print("QUESTION 3:", question_3)
print("=" * 80)

# new
context_user = context.copy()
print("\nNEW PROMPT OUTPUT:")
print(return_CCRMSQL(question_3, context_user))

# old
old_context_user = old_context.copy()
print("\nOLD PROMPT OUTPUT:")
print(return_CCRMSQL(question_3, old_context_user))


print("\n" + "=" * 80)
print("QUESTION 4:", question_4)
print("=" * 80)

# new
context_user = context.copy()
print("\nNEW PROMPT OUTPUT:")
print(return_CCRMSQL(question_4, context_user))

# old
old_context_user = old_context.copy()
print("\nOLD PROMPT OUTPUT:")
print(return_CCRMSQL(question_4, old_context_user))

QUESTION 1: Who is the best paid employee?

NEW PROMPT OUTPUT:
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;

OLD PROMPT OUTPUT:
This is your SQL:
```sql
SELECT e.name, s.salary
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;
```

QUESTION 2: What is the average salary by institution?

NEW PROMPT OUTPUT:
SELECT st.Institution, AVG(sa.salary) AS avg_salary
FROM studies st
JOIN salary sa ON st.ID_usr = sa.ID_usr
GROUP BY st.Institution
ORDER BY avg_salary DESC;

OLD PROMPT OUTPUT:
This is your SQL:
```sql
SELECT s.Institution, AVG(s.salary) AS average_salary
FROM salary s
INNER JOIN employees e ON s.ID_usr = e.ID_usr
GROUP BY s.Institution;
```

QUESTION 3: List the employees and their specialities.

NEW PROMPT OUTPUT:
```sql
SELECT e.name, st.Speciality
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr;
```

OLD PROMPT OUTPUT:
This is your SQL:
```sql
SELECT e.name, s.Speciality
FROM emplo

Final Report — NL2SQL Prompt Comparison

In this lab, I compared two different prompt strategies for generating SQL queries from natural language questions.

The old prompt describes the database using a JSON-like structure. It provides the table names and fields, but it gives less guidance to the model. Because of this, the model can sometimes return extra explanations or generate less precise SQL.

The new prompt uses SQL-style table definitions and few-shot examples. This gives the model a clearer idea of the database structure and of the expected answer format.

I tested both prompts with questions about employees, salaries, and studies. The new prompt produced better and more direct SQL queries, especially when joins were needed between different tables.

The main conclusion is that prompt design has a strong impact on NL2SQL results. A clearer prompt with table structure and example queries helps the model generate more accurate SQL.